# Module 1: Data Loading and Augmentation Using Keras

This notebook completes the required tasks for Keras-based data loading and augmentation.
It collects image paths, creates a custom generator, and builds validation data.

In [1]:
import os
import random
import numpy as np
from PIL import Image
import tensorflow as tf

base_dir = './images_dataSAT'
if not os.path.exists(base_dir):
    alt_dir = os.path.join('.', 'AI Capstone DL Projects', 'CNN Model Development', 'images_dataSAT')
    if os.path.exists(alt_dir):
        base_dir = alt_dir

print('Dataset directory:', base_dir)

dir_non_agri = os.path.join(base_dir, 'class_0_non_agri')
dir_agri = os.path.join(base_dir, 'class_1_agri')

print('Non-agri folder exists:', os.path.exists(dir_non_agri))
print('Agri folder exists:', os.path.exists(dir_agri))

c:\Users\Furqan Khan\AppData\Local\miniconda3\envs\agent_env2\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Dataset directory: .\AI Capstone DL Projects\CNN Model Development\images_dataSAT
Non-agri folder exists: True
Agri folder exists: True


## Task 1: Create all_image_paths

This list contains all image file paths from both classes in the base directory.

In [2]:
all_image_paths = []
all_labels = []

for folder, label in [(dir_non_agri, 0), (dir_agri, 1)]:
    for filename in sorted(os.listdir(folder)):
        all_image_paths.append(os.path.join(folder, filename))
        all_labels.append(label)

print('Total image paths:', len(all_image_paths))
print('First 5 paths:')
for path in all_image_paths[:5]:
    print(path)

Total image paths: 6000
First 5 paths:
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_0_non_agri\tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_5902.jpg
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_0_non_agri\tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_6074.jpg
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_0_non_agri\tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_6246.jpg
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_0_non_agri\tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_6247.jpg
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_0_non_agri\tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_6248.jpg


## Task 2: Create temp and print 5 random pairs

The zip function binds each image path to its label.
Then we randomly select and print 5 samples.

In [3]:
temp = list(zip(all_image_paths, all_labels))
random.shuffle(temp)
random_samples = temp[:5]

print('Random sample pairs:')
for image_path, label in random_samples:
    print(image_path, '->', label)

Random sample pairs:
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_1_agri\tile_S2A_MSIL2A_20250427T101701_N0511_R065_T32UPE_20250427T170513.SAFE_15069.jpg -> 1
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_1_agri\tile_S2A_MSIL2A_20250427T101701_N0511_R065_T32UPE_20250427T170513.SAFE_12465.jpg -> 1
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_0_non_agri\tile_S2A_MSIL2A_20250427T101701_N0511_R065_T32UQB_20250427T170513.SAFE_16981.jpg -> 0
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_1_agri\tile_S2A_MSIL2A_20250427T101701_N0511_R065_T32UQB_20250427T170513.SAFE_3937.jpg -> 1
.\AI Capstone DL Projects\CNN Model Development\images_dataSAT\class_0_non_agri\tile_S2A_MSIL2A_20250427T101701_N0511_R065_T32UPE_20250427T170513.SAFE_1194.jpg -> 0


## Task 3: Create a custom data generator

This generator yields image batches of size 8.

In [4]:
def custom_data_generator(image_paths, labels, batch_size=8):
    while True:
        batch_images = []
        batch_labels = []

        for image_path, label in zip(image_paths, labels):
            image = Image.open(image_path).convert('RGB')
            image = image.resize((64, 64))
            image_array = np.array(image, dtype=np.float32) / 255.0
            batch_images.append(image_array)
            batch_labels.append(label)

            if len(batch_images) == batch_size:
                yield np.array(batch_images), np.array(batch_labels)
                batch_images = []
                batch_labels = []

        if len(batch_images) > 0:
            yield np.array(batch_images), np.array(batch_labels)

train_generator = custom_data_generator(all_image_paths, all_labels, batch_size=8)
batch_images, batch_labels = next(train_generator)

print('Batch image shape:', batch_images.shape)
print('Batch label shape:', batch_labels.shape)
print('Batch labels:', batch_labels)

Batch image shape: (8, 64, 64, 3)
Batch label shape: (8,)
Batch labels: [0 0 0 0 0 0 0 0]


## Task 4: Create validation data

We split the dataset and generate validation batches with size 8.

In [5]:
validation_split = 0.2
validation_count = int(len(all_image_paths) * validation_split)

validation_paths = all_image_paths[:validation_count]
validation_labels = all_labels[:validation_count]

validation_data = custom_data_generator(validation_paths, validation_labels, batch_size=8)
val_images, val_labels = next(validation_data)

print('Validation batch image shape:', val_images.shape)
print('Validation batch label shape:', val_labels.shape)
print('Validation labels:', val_labels)

Validation batch image shape: (8, 64, 64, 3)
Validation batch label shape: (8,)
Validation labels: [0 0 0 0 0 0 0 0]
